<a href="https://colab.research.google.com/github/bingjunw/ust-deep-learning-2026/blob/main/Assignment_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 4

**Load Libraries and Set Up Data**

In [23]:
# Import necessary libraries
import os
import time
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split

# Set up data
# Clear Keras session to release memory and reset layer naming
tf.keras.backend.clear_session()
# Set the random seed
SEED = 2026
tf.random.set_seed(SEED)
np.random.seed(SEED)

L = 400  # Size
num_samples = 100 # Number of samples

# Download the data
!wget --no-check-certificate 'https://drive.google.com/uc?export=download&id=1QbOSExVJEbPMhjzaua5n2eIXeF3qELQ7' -O "label.txt"
!pip install gdown
!gdown --id '1Sh2ce0jo5FVGNsSa9fqLjqcAOWQBFhzz' -O "encoded_seq.txt"
# Reformat the input data
def load_data():
    labels = np.loadtxt('label.txt')
    encoded_seq = np.loadtxt('encoded_seq.txt')
    encoded_seq_choose = encoded_seq[:, ((400-L)*2):(1600-(400-L)*2)]
    print(encoded_seq_choose.shape)
    # Reshape the data to match the model's expected input shape (L, 4)
    encoded_seq_choose = encoded_seq_choose.reshape(-1, L, 4)
    print(encoded_seq_choose.shape)
    x_train,x_test,y_train,y_test = train_test_split(encoded_seq_choose,labels,test_size=0.2)

    # One-hot encode the labels
    num_classes = 3 # Assuming 3 classes based on model's last layer output
    y_train_one_hot = tf.keras.utils.to_categorical(y_train, num_classes=num_classes)
    y_test_one_hot = tf.keras.utils.to_categorical(y_test, num_classes=num_classes)

    return np.array(x_train), np.array(y_train_one_hot), np.array(x_test), np.array(y_test_one_hot)

x_train,y_train,x_test,y_test = load_data()

--2026-02-10 02:07:41--  https://drive.google.com/uc?export=download&id=1QbOSExVJEbPMhjzaua5n2eIXeF3qELQ7
Resolving drive.google.com (drive.google.com)... 173.194.79.101, 173.194.79.113, 173.194.79.100, ...
Connecting to drive.google.com (drive.google.com)|173.194.79.101|:443... connected.
HTTP request sent, awaiting response... 303 See Other
Location: https://drive.usercontent.google.com/download?id=1QbOSExVJEbPMhjzaua5n2eIXeF3qELQ7&export=download [following]
--2026-02-10 02:07:41--  https://drive.usercontent.google.com/download?id=1QbOSExVJEbPMhjzaua5n2eIXeF3qELQ7&export=download
Resolving drive.usercontent.google.com (drive.usercontent.google.com)... 108.177.96.132, 2a00:1450:4013:c06::84
Connecting to drive.usercontent.google.com (drive.usercontent.google.com)|108.177.96.132|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 60000 (59K) [application/octet-stream]
Saving to: ‘label.txt’

label.txt           100%[===================>]  58.59K  --.-KB/s    in 0

**Define the Model Architectures**

In [24]:
# Create the splice finder model
def create_sfm():
    model = models.Sequential()
    model.add(layers.Input(shape=(L, 4)))
    model.add(layers.Conv1D(filters=50, kernel_size=5, activation='relu'))
    model.add(layers.Flatten())
    model.add(layers.Dense(100, activation='relu'))
    model.add(layers.Dense(3, activation='softmax'))
    return model

splice_finder_model = create_sfm()
splice_finder_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 396, 50)        │         1,050 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 19800)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 100)            │     1,980,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │           303 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,981,453 (7.56 MB)

 Trainable params: 1,981,453 (7.56 MB)

 Non-trainable params: 0 (0.00 B)

**Training loop on CPU**



In [25]:
# Add early_stop to reduce the time
early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

# Train the model on CPU
with tf.device("/CPU:0"):
    model = create_sfm()
    model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])
    start_time = time.time()
    model.fit(
        x_train, y_train,
        validation_data=(x_test, y_test),
        epochs=50,
        batch_size=32,
        callbacks=[early_stop],
        verbose=1
    )
    cpu_time = time.time() - start_time

Epoch 1/50
750/750 ━━━━━━━━━━━━━━━━━━━━ 24s 30ms/step - accuracy: 0.8351 - loss: 0.4267 - val_accuracy: 0.9625 - val_loss: 0.1297
Epoch 2/50
750/750 ━━━━━━━━━━━━━━━━━━━━ 21s 28ms/step - accuracy: 0.9689 - loss: 0.1011 - val_accuracy: 0.9630 - val_loss: 0.1272
Epoch 3/50
750/750 ━━━━━━━━━━━━━━━━━━━━ 43s 30ms/step - accuracy: 0.9853 - loss: 0.0524 - val_accuracy: 0.9605 - val_loss: 0.1570
Epoch 4/50
750/750 ━━━━━━━━━━━━━━━━━━━━ 39s 27ms/step - accuracy: 0.9895 - loss: 0.0335 - val_accuracy: 0.9668 - val_loss: 0.1457
Epoch 5/50
750/750 ━━━━━━━━━━━━━━━━━━━━ 21s 28ms/step - accuracy: 0.9921 - loss: 0.0216 - val_accuracy: 0.9678 - val_loss: 0.1625


**Training loop on GPU**

In [26]:
# Train the model on GPU
if tf.config.list_physical_devices('GPU'):
  with tf.device("/GPU:0"):
        model = create_sfm()
        model.compile(optimizer='adam',
                      loss='categorical_crossentropy',
                      metrics=['accuracy'])

        start_time = time.time()
        model.fit(
            x_train, y_train,
            validation_data=(x_test, y_test),
            epochs=50,
            batch_size=32,
            callbacks=[early_stop],
            verbose=1
        )
        gpu_time = time.time() - start_time
else:
    gpu_time = 0
    print("GPU not found.")

Epoch 1/50
750/750 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - accuracy: 0.8451 - loss: 0.3877 - val_accuracy: 0.9587 - val_loss: 0.1297
Epoch 2/50
750/750 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.9728 - loss: 0.0879 - val_accuracy: 0.9627 - val_loss: 0.1277
Epoch 3/50
750/750 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.9882 - loss: 0.0435 - val_accuracy: 0.9638 - val_loss: 0.1406
Epoch 4/50
750/750 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9921 - loss: 0.0246 - val_accuracy: 0.9680 - val_loss: 0.1373
Epoch 5/50
750/750 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.9945 - loss: 0.0161 - val_accuracy: 0.9623 - val_loss: 0.1824


**Conclusion**


In [27]:
# Visualize the result
print(f"CPU Running time: {cpu_time:.2f}s")
print(f"GPU Running time: {gpu_time:.2f}s")
if gpu_time > 0:
    speedup = cpu_time / gpu_time
    print(f"GPU is {speedup:.2f} time faster than CPU")


CPU Running time: 147.45s
GPU Running time: 18.83s
GPU is 7.83 time faster than CPU
